### Capa Silver (Limpieza y Estandarización)
Esta capa procesa los datos crudos de la capa Bronze aplicando reglas de calidad y transformaciones, para luego almacenarlos en formato Delta Lake. Las operaciones principales incluyen:

- **Filtrado y validación:** Eliminación de registros con claves primarias nulas (PK_Transaccion) y transacciones con monto cero o negativo.
- **Limpieza numérica:** Manejo de separadores decimales (reemplazo de comas por puntos) y conversión a tipos numéricos correctos (enteros/flotantes).
- **Deduplicación:** Eliminación de registros transaccionales duplicados basados en su clave principal.
- **Estandarización de texto:** Eliminación de espacios innecesarios (	rim) y conversión a mayúsculas para asegurar consistencia (ej. códigos de Moneda, FK_Canal).
- **Upsert en Delta Lake:** Almacenamiento y actualización incremental utilizando operaciones MERGE dentro de la tabla gestionada en Databricks: **	ransacciones_plata_robert**.

In [0]:

from pyspark.sql.functions import col, trim, upper, round, regexp_replace,coalesce, to_timestamp,to_date, date_format, try_to_timestamp, expr
from delta.tables import DeltaTable



# Declaración de lectura
# Los datos fueron inyectados directamente por GitHub Actions
# Lectura desde el Lakehouse (Unity Catalog Volume)
df_extraido = (spark.read
  .format("csv")
  .option("header", "true")
  .option("inferSchema", "true")
  .load("/Volumes/main/aml_proyect/raw_data/transacciones_aml_*.csv")
)

# Descarga forzada al almacenamiento interno de Databricks
df_extraido.write   \
  .format("delta")  \
  .option("overwriteSchema", "true") \
  .mode("overwrite") \
  .saveAsTable("bronce_azure_local")

# Lectura de datos
df_bronce_local = spark.table("bronce_azure_local")

# Limpieza de datos
print("2. Aplicando reglas de calidad (Capa Plata)...")
df_silver = (df_bronce_local
    .filter(col("PK_Transaccion").isNotNull())
    .withColumn("Monto_Original", round(col("Monto_Original").cast("double"), 2))
    .withColumn("Monto_USD",round(col("Monto_USD").cast("double"), 2))
    .filter(col("Monto_USD") > 0)
    .dropDuplicates(["PK_Transaccion"])
    .withColumn("FK_Canal", trim(col("FK_Canal")))
    .withColumn("Moneda", upper(trim(col("Moneda"))))
    .withColumn("Tiempo_Real", expr("coalesce(try_to_timestamp(FK_Tiempo, 'yyyy-MM-dd HH:mm:ss'), try_to_timestamp(FK_Tiempo, 'dd/MM/yyyy HH:mm:ss'))"))
    .withColumn("Fecha_Transaccion", to_date(col("Tiempo_Real")))
    .withColumn("Hora_Transaccion", date_format(col("Tiempo_Real"), "HH:mm:ss"))
    .drop("FK_Tiempo", "Tiempo_Real")
)




# Guardar la tabla

print("3. Ejecutando MERGE (Upsert) en tabla Delta...")

# Conectamos con la tabla final que ya creaste en la ejecución anterior
tabla_destino = DeltaTable.forName(spark, "aml_proyect.transacciones_plata_robert")

# Hacemos el cruce inteligente
tabla_destino.alias("destino").merge(
    df_silver.alias("origen"),
    "destino.PK_Transaccion = origen.PK_Transaccion"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

# Mostramos el resultado
display(spark.sql("SELECT * FROM aml_proyect.transacciones_plata_robert"))